# Train ST-GCN++ for Badminton Actions

In Colab, select **Runtime > Change runtime type > GPU** before running. This is an exploratory baseline because all match clips currently come from one source match.

In [ ]:
!nvidia-smi
!git clone https://github.com/dattt-cy/Badminton_AI.git /content/Badminton_AI
%cd /content/Badminton_AI

Upload the generated `data/annotations/badminton_actions.pkl` file from your computer. Raw videos are not needed for training.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()
source = Path(next(iter(uploaded)))
destination = Path('/content/Badminton_AI/data/annotations/badminton_actions.pkl')
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(source), destination)
print(destination, destination.stat().st_size)

Run the next cell once. Colab will restart the runtime after installing Conda; reconnect, return to this notebook, and continue with the following cell.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
%cd /content/Badminton_AI
# Use the classic solver and a minimal environment. The full legacy PySKL
# environment file can crash libmamba on current Colab runtimes.
!CONDA_SOLVER=classic conda create -y -n pyskl_310 python=3.10 pip
!conda run -n pyskl_310 python -m pip install --no-cache-dir numpy==1.23.5 scipy==1.9.3 pyyaml tqdm addict yapf==0.32.0 packaging termcolor opencv-python-headless==4.7.0.72
!conda run -n pyskl_310 python -m pip install --no-cache-dir torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113
!conda run -n pyskl_310 python -m pip install --no-cache-dir mmcv-full==1.7.0 -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.12.0/index.html
!conda run -n pyskl_310 python -m pip install --no-deps -e external/pyskl
!conda run -n pyskl_310 python -m pip install --no-deps -e .

In [ ]:
!conda run -n pyskl_310 python -c "import torch, mmcv, pyskl; print('torch', torch.__version__, 'mmcv', mmcv.__version__, 'cuda', torch.cuda.is_available())"

Train and validate. Rotation (±6.9 degrees) and scaling (±10%) are generated dynamically from skeletons during training.

In [ ]:
!conda run -n pyskl_310 python external/pyskl/tools/train.py configs/action_recognition/stgcnpp_badminton.py --validate --test-best

In [ ]:
from google.colab import files
!zip -qr /content/stgcnpp_badminton_results.zip work_dirs/stgcnpp_badminton
files.download('/content/stgcnpp_badminton_results.zip')